In [1]:
# 데이터 확인하기 2025.11.20
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")  # 모든 경고 무시
from matplotlib import rc
plt.rc('font', family='Malgun Gothic')

from utils import get_clf_eval

In [2]:
# 데이터 로딩
data_path_train = '../data/train.csv'
data_path_test  = '../data/test.csv'

train = pd.read_csv(data_path_train)
test  = pd.read_csv(data_path_test)

In [3]:
# Data 전처리 1. zero_count_rate이 99%인 컬럼 제거하기 
remove_cols = pd.read_csv('../doc/remove_cols.xls', header=0).squeeze() 
train.drop(columns=remove_cols, axis=1, inplace=True)
test.drop(columns=remove_cols, axis=1, inplace=True)

In [4]:
# 데이터 할당
X_features = train.drop(columns=['ID', 'TARGET'], axis=1) # ID와 TARGET 모두 제거하고 X_train 만들기
y_labels   = train['TARGET']
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 
X_features.columns # 149개

Index(['var3', 'var15', 'imp_ent_var16_ult1', 'imp_op_var39_comer_ult1',
       'imp_op_var39_comer_ult3', 'imp_op_var41_comer_ult1',
       'imp_op_var41_comer_ult3', 'imp_op_var41_efect_ult1',
       'imp_op_var41_efect_ult3', 'imp_op_var41_ult1',
       ...
       'saldo_medio_var8_ult3', 'saldo_medio_var12_hace2',
       'saldo_medio_var12_hace3', 'saldo_medio_var12_ult1',
       'saldo_medio_var12_ult3', 'saldo_medio_var13_corto_hace2',
       'saldo_medio_var13_corto_hace3', 'saldo_medio_var13_corto_ult1',
       'saldo_medio_var13_corto_ult3', 'var38'],
      dtype='object', length=149)

In [5]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [6]:
# 스케일링
# X_train_scaled, X_test_scaled, scaler = scale_data(X_train=X, X_test=X_test)
scaler        = StandardScaler()
X_scaled      = scaler.fit_transform(X_features)
X_test_scaled = scaler.transform(X_test)

In [7]:
# 레이블의 분포 확인
cust_cnt = y_labels.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [8]:
type(X_scaled)

numpy.ndarray

In [9]:
# first testing model
# XGBoost (xgb) : yjh, kjh
# LightGBM(lgbm) : lsj, ujm
# Random Forest(rf) : lkj, kjh
# Logistic Regression(lr) : yjh, ujm


In [10]:
# 학습/테스트 데이터 분리
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
  X_features, 
  y_labels,
  test_size    = 0.2, # 8:2
  random_state = 23, # 세미프로젝트3조
  stratify = y_labels
)


In [11]:
# Logistic Regression 

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 모델 생성 및 학습
log_reg = LogisticRegression(max_iter=1000, random_state=23)
log_reg.fit(X_train, y_train)

# 예측
y_pred = log_reg.predict(X_val)
pred_proba = log_reg.predict_proba(X_val)[:,1]

# 평가
print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, pred_proba))

# Accuracy: 0.9604051565377533
# ROC-AUC: 0.6228827480511704

Accuracy: 0.9604051565377533
ROC-AUC: 0.6228827480511704


In [12]:
# XGBoost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=23
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_val)
pred_proba = xgb.predict_proba(X_val)[:,1]

print("Accuracy:", accuracy_score(y_val, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_val, pred_proba))

# Accuracy: 0.9606024730334123
# ROC-AUC: 0.841652897864535

Accuracy: 0.9606024730334123
ROC-AUC: 0.841652897864535
